# Short-List Promising Models & Fine-Tune the System

In [35]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from custom_transformers import *

In [36]:
train_set = pd.read_parquet("train_set.parquet")
y = train_set["diseased"]
X = train_set.drop(columns="diseased")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [37]:
log_reg = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
log_reg_recall = cross_val_score(log_reg, X, y, cv=cv, scoring="recall")
log_reg_precision = cross_val_score(log_reg, X, y, cv=cv, scoring="precision")
log_reg_f1 = cross_val_score(log_reg, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": log_reg_recall,
    "precision": log_reg_precision,
    "f1": log_reg_f1,
}).mean().round(3)

recall       0.749
precision    0.428
f1           0.543
dtype: float64

Logistic regression performed well. Across the 5 folds it found about 75% of the sick sightings with a recall of 0.749, and about 43% of the sightings it flagged were actually sick with a precision of 0.428. The F1 score was 0.543. Recall was a better score than precision and f1.

In [38]:
sgd = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)),
])
sgd_recall = cross_val_score(sgd, X, y, cv=cv, scoring="recall")
sgd_precision = cross_val_score(sgd, X, y, cv=cv, scoring="precision")
sgd_f1 = cross_val_score(sgd, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": sgd_recall,
    "precision": sgd_precision,
    "f1": sgd_f1,
}).mean().round(3)

recall       0.785
precision    0.303
f1           0.425
dtype: float64

The SGD classifier had the highest recall of the six models at 0.785, but its precision dropped to 0.303, so most of the sightings it flagged were healthy. Its F1 score was 0.425.

In [39]:
svc = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", SVC(class_weight="balanced", random_state=42)),
])
svc_recall = cross_val_score(svc, X, y, cv=cv, scoring="recall")
svc_precision = cross_val_score(svc, X, y, cv=cv, scoring="precision")
svc_f1 = cross_val_score(svc, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": svc_recall,
    "precision": svc_precision,
    "f1": svc_f1,
}).mean().round(3)

recall       0.716
precision    0.448
f1           0.551
dtype: float64

The SVC performed well and was very close to logistic regression. It found about 72% of the sick sightings with a recall of 0.716 and had a precision of 0.448, giving an F1 score of 0.551.

In [40]:
tree = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
])
tree_recall = cross_val_score(tree, X, y, cv=cv, scoring="recall")
tree_precision = cross_val_score(tree, X, y, cv=cv, scoring="precision")
tree_f1 = cross_val_score(tree, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": tree_recall,
    "precision": tree_precision,
    "f1": tree_f1,
}).mean().round(3)

recall       0.442
precision    0.438
f1           0.440
dtype: float64

The decision tree was the weakest model. It found only 44% of the sick sightings with a recall of 0.442, the lowest of the six, and had a precision of 0.438 and an F1 score of 0.440. A single tree with no depth limit overfits the training folds.

In [41]:
forest = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42)),
])
forest_recall = cross_val_score(forest, X, y, cv=cv, scoring="recall")
forest_precision = cross_val_score(forest, X, y, cv=cv, scoring="precision")
forest_f1 = cross_val_score(forest, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": forest_recall,
    "precision": forest_precision,
    "f1": forest_f1,
}).mean().round(3)

recall       0.617
precision    0.546
f1           0.576
dtype: float64

The random forest performed well and had the highest F1 score of the six models at 0.576. Its precision of 0.546 was better than the linear models, but it found fewer sick sightings with a recall of 0.617.

In [42]:
boost = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", GradientBoostingClassifier(random_state=42)),
])
boost_recall = cross_val_score(boost, X, y, cv=cv, scoring="recall")
boost_precision = cross_val_score(boost, X, y, cv=cv, scoring="precision")
boost_f1 = cross_val_score(boost, X, y, cv=cv, scoring="f1")

pd.DataFrame({
    "recall": boost_recall,
    "precision": boost_precision,
    "f1": boost_f1,
}).mean().round(3)

recall       0.466
precision    0.679
f1           0.549
dtype: float64

Gradient boosting had the highest precision at 0.679, so most of the sightings it flagged were actually sick. Its recall was lower at 0.466 because it does not support class weights, so it leans toward predicting healthy. Its F1 score was 0.549, close to the SVC.